# Grad-CAM Analyse
Baseline vs. Augmentation – Testdatensatz
Vgl. Kapitel 4.5 und 5.4

In [ ]:
import sys
sys.path.append('..')

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from config import DATA, PATHS
from data.loader import load_test_dataset
from evaluation.grad_cam import (
    compute_grad_cam,
    overlay_heatmap,
    classify_test_images,
    plot_grad_cam_row,
    LABEL_MAP,
    LAYER_NAME
)

CLASS_NAMES = DATA['classes']
print('Setup abgeschlossen.')

## Zelle 2: Modelle + Testdatensatz laden

In [ ]:
baseline     = tf.keras.models.load_model(str(PATHS['models'] / 'baseline.keras'))
augmentation = tf.keras.models.load_model(str(PATHS['models'] / 'augmentation.keras'))

print('Baseline geladen:     ', str(PATHS['models'] / 'baseline.keras'))
print('Augmentation geladen: ', str(PATHS['models'] / 'augmentation.keras'))

test_ds = load_test_dataset(batch_size=1)
print(f'Testbilder: {len(list(test_ds))}')

In [ ]:
base_model = baseline.get_layer('mobilenetv2_1.00_224')
layer = base_model.get_layer(LAYER_NAME)

print(f"Verwendete Grad-CAM Schicht : {LAYER_NAME}")
print(f"Layer-Typ                   : {layer.__class__.__name__}")
print(f"Output-Shape                : {layer.output_shape}")

## Zelle 3: Alle Testbilder klassifizieren

In [ ]:
print('Klassifiziere mit Baseline...')
images_list, y_true_b, y_pred_b, conf_b = classify_test_images(baseline, test_ds)

print('Klassifiziere mit Augmentation...')
_, y_true_a, y_pred_a, conf_a = classify_test_images(augmentation, test_ds)

print(f'\nBaseline     – Accuracy: {np.mean(y_true_b == y_pred_b):.4f}')
print(f'Augmentation – Accuracy: {np.mean(y_true_a == y_pred_a):.4f}')

wrong_b    = np.where(y_true_b != y_pred_b)[0]
wrong_a    = np.where(y_true_a != y_pred_a)[0]
low_conf_b = np.argsort(conf_b)[:5]
low_conf_a = np.argsort(conf_a)[:5]

print(f'\nFehlklassifikationen Baseline:     {len(wrong_b)}')
print(f'Fehlklassifikationen Augmentation: {len(wrong_a)}')

## Zelle 4: Korrekte Klassifikationen – Baseline vs. Augmentation
2 Bilder je Klasse, beide Modelle nebeneinander

In [ ]:
for class_idx, class_name in enumerate(CLASS_NAMES):
    correct_idx = np.where(
        (y_true_b == class_idx) & (y_pred_b == class_idx)
    )[0][:2]

    if len(correct_idx) == 0:
        print(f'Keine korrekten Klassifikationen für {class_name}')
        continue

    for idx in correct_idx:
        image = images_list[idx]

        heatmap_b = compute_grad_cam(baseline,     image, class_idx)
        heatmap_a = compute_grad_cam(augmentation, image, class_idx)

        true_display   = LABEL_MAP.get(class_name, class_name)
        pred_display_b = LABEL_MAP.get(CLASS_NAMES[y_pred_b[idx]], CLASS_NAMES[y_pred_b[idx]])
        pred_display_a = LABEL_MAP.get(CLASS_NAMES[y_pred_a[idx]], CLASS_NAMES[y_pred_a[idx]])

        fig, axes = plt.subplots(2, 2, figsize=(10, 8))
        fig.suptitle(
            f'Wahre Klasse: {true_display} | '
            f'Baseline (oben): {pred_display_b} ({conf_b[idx]:.2%}) | '
            f'Augmentation (unten): {pred_display_a} ({conf_a[idx]:.2%})',
            fontsize=10
        )

        plot_grad_cam_row(
            axes[0], image, heatmap_b,
            CLASS_NAMES[y_true_b[idx]], CLASS_NAMES[y_pred_b[idx]],
            conf_b[idx], model_name='Baseline'
        )
        plot_grad_cam_row(
            axes[1], image, heatmap_a,
            CLASS_NAMES[y_true_a[idx]], CLASS_NAMES[y_pred_a[idx]],
            conf_a[idx], model_name='Augmentation'
        )

        plt.tight_layout(rect=[0, 0, 1, 0.92])
        output_path = PATHS['grad_cam'] / f'correct_{class_name}_{idx}.png'
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'Gespeichert: {output_path}')

## Zelle 5: Fehlklassifikationen – alle
Baseline und Augmentation getrennt

In [ ]:
for model_name, wrong, y_true, y_pred, conf, model in [
    ('Baseline',     wrong_b, y_true_b, y_pred_b, conf_b, baseline),
    ('Augmentation', wrong_a, y_true_a, y_pred_a, conf_a, augmentation)
]:
    if len(wrong) == 0:
        print(f'{model_name}: keine Fehlklassifikationen')
        continue

    print(f'\n{model_name} – {len(wrong)} Fehlklassifikationen:')

    fig, axes = plt.subplots(len(wrong), 2, figsize=(10, 4 * len(wrong)))
    if len(wrong) == 1:
        axes = axes[np.newaxis, :]

    fig.suptitle(f'Fehlklassifikationen – {model_name}', fontsize=13)

    for row, idx in enumerate(wrong):
        image   = images_list[idx]
        heatmap = compute_grad_cam(model, image, y_pred[idx])

        true_display = LABEL_MAP.get(CLASS_NAMES[y_true[idx]], CLASS_NAMES[y_true[idx]])
        pred_display = LABEL_MAP.get(CLASS_NAMES[y_pred[idx]], CLASS_NAMES[y_pred[idx]])

        plot_grad_cam_row(
            axes[row], image, heatmap,
            CLASS_NAMES[y_true[idx]], CLASS_NAMES[y_pred[idx]],
            conf[idx], model_name=model_name
        )
        print(f'  Index {idx}: Wahr={true_display} Pred={pred_display} Konfidenz={conf[idx]:.4f}')

    plt.tight_layout(rect=[0, 0, 1, 0.97])
    output_path = PATHS['grad_cam'] / f'misclassified_{model_name.lower()}.png'
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Gespeichert: {output_path}')

## Zelle 6: Grenzfälle – niedrige Konfidenz

In [ ]:
for model_name, low_conf_idx, y_true, y_pred, conf, model in [
    ('Baseline',     low_conf_b, y_true_b, y_pred_b, conf_b, baseline),
    ('Augmentation', low_conf_a, y_true_a, y_pred_a, conf_a, augmentation)
]:
    print(f'\n{model_name} – Grenzfälle (niedrigste Konfidenz):')

    fig, axes = plt.subplots(len(low_conf_idx), 2, figsize=(10, 4 * len(low_conf_idx)))
    if len(low_conf_idx) == 1:
        axes = axes[np.newaxis, :]

    fig.suptitle(f'Grenzfälle – {model_name}', fontsize=13)

    for row, idx in enumerate(low_conf_idx):
        image   = images_list[idx]
        heatmap = compute_grad_cam(model, image, y_pred[idx])

        true_display = LABEL_MAP.get(CLASS_NAMES[y_true[idx]], CLASS_NAMES[y_true[idx]])
        pred_display = LABEL_MAP.get(CLASS_NAMES[y_pred[idx]], CLASS_NAMES[y_pred[idx]])

        plot_grad_cam_row(
            axes[row], image, heatmap,
            CLASS_NAMES[y_true[idx]], CLASS_NAMES[y_pred[idx]],
            conf[idx], model_name=model_name
        )
        print(f'  Index {idx}: Wahr={true_display} Pred={pred_display} Konfidenz={conf[idx]:.4f}')

    plt.tight_layout(rect=[0, 0, 1, 0.97])
    output_path = PATHS['grad_cam'] / f'low_confidence_{model_name.lower()}.png'
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Gespeichert: {output_path}')

## Zelle 7: Direkter Vergleich – dasselbe Bild durch beide Modelle
Wähle einen Index manuell aus den obigen Ergebnissen

In [ ]:
# ── Index hier manuell setzen ─────────────────────────────
IDX = 0    # anpassen nach Analyse der obigen Zellen
# ─────────────────────────────────────────────────────────

image = images_list[IDX]

heatmap_b = compute_grad_cam(baseline,     image, y_pred_b[IDX])
heatmap_a = compute_grad_cam(augmentation, image, y_pred_a[IDX])

true_display   = LABEL_MAP.get(CLASS_NAMES[y_true_b[IDX]], CLASS_NAMES[y_true_b[IDX]])
pred_display_b = LABEL_MAP.get(CLASS_NAMES[y_pred_b[IDX]], CLASS_NAMES[y_pred_b[IDX]])
pred_display_a = LABEL_MAP.get(CLASS_NAMES[y_pred_a[IDX]], CLASS_NAMES[y_pred_a[IDX]])

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
fig.suptitle(
    f'Wahre Klasse: {true_display} | '
    f'Baseline (oben): {pred_display_b} ({conf_b[IDX]:.2%}) | '
    f'Augmentation (unten): {pred_display_a} ({conf_a[IDX]:.2%})',
    fontsize=10
)

plot_grad_cam_row(
    axes[0], image, heatmap_b,
    CLASS_NAMES[y_true_b[IDX]], CLASS_NAMES[y_pred_b[IDX]],
    conf_b[IDX], model_name='Baseline'
)
plot_grad_cam_row(
    axes[1], image, heatmap_a,
    CLASS_NAMES[y_true_a[IDX]], CLASS_NAMES[y_pred_a[IDX]],
    conf_a[IDX], model_name='Augmentation'
)

plt.tight_layout(rect=[0, 0, 1, 0.92])
output_path = PATHS['grad_cam'] / f'comparison_idx{IDX}.png'
plt.savefig(output_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Gespeichert: {output_path}')

## Zelle 8: GoPro-Positions-Analyse

In [ ]:
test_ds_with_paths = tf.keras.utils.image_dataset_from_directory(
    PATHS['dataset_test'],
    image_size=DATA['img_size'],
    batch_size=1,
    shuffle=False,
    labels='inferred',
    label_mode='int'
)

file_paths = test_ds_with_paths.file_paths

def get_position(path):
    name = Path(path).stem.lower()
    if 'oben' in name:
        return 'oben'
    elif 'mitte' in name:
        return 'mitte'
    elif 'unten' in name:
        return 'unten'
    return 'unknown'

positions = [get_position(p) for p in file_paths]

pos_indices = {
    'oben':  [i for i, p in enumerate(positions) if p == 'oben'],
    'mitte': [i for i, p in enumerate(positions) if p == 'mitte'],
    'unten': [i for i, p in enumerate(positions) if p == 'unten'],
}

print('Bilder je Position:')
for pos, idxs in pos_indices.items():
    print(f'  {pos}: {len(idxs)} Bilder')

In [ ]:
# Metriken je Position
for pos, idxs in pos_indices.items():
    if not idxs:
        continue

    acc_b = np.mean(y_true_b[idxs] == y_pred_b[idxs])
    acc_a = np.mean(y_true_a[idxs] == y_pred_a[idxs])

    print(f'\nPosition: {pos} (n={len(idxs)})')
    print(f'  Baseline     Accuracy: {acc_b:.4f}')
    print(f'  Augmentation Accuracy: {acc_a:.4f}')

In [ ]:
# Grad-CAM je Position
for pos, idxs in pos_indices.items():
    if not idxs:
        continue

    idx   = idxs[0]
    image = images_list[idx]

    heatmap_b = compute_grad_cam(baseline,     image, y_pred_b[idx])
    heatmap_a = compute_grad_cam(augmentation, image, y_pred_a[idx])

    true_display   = LABEL_MAP.get(CLASS_NAMES[y_true_b[idx]], CLASS_NAMES[y_true_b[idx]])
    pred_display_b = LABEL_MAP.get(CLASS_NAMES[y_pred_b[idx]], CLASS_NAMES[y_pred_b[idx]])
    pred_display_a = LABEL_MAP.get(CLASS_NAMES[y_pred_a[idx]], CLASS_NAMES[y_pred_a[idx]])

    fig, axes = plt.subplots(2, 2, figsize=(10, 8))
    fig.suptitle(
        f'GoPro Position: {pos} | Wahre Klasse: {true_display} | '
        f'Baseline (oben): {pred_display_b} ({conf_b[idx]:.2%}) | '
        f'Augmentation (unten): {pred_display_a} ({conf_a[idx]:.2%})',
        fontsize=9
    )

    plot_grad_cam_row(
        axes[0], image, heatmap_b,
        CLASS_NAMES[y_true_b[idx]], CLASS_NAMES[y_pred_b[idx]],
        conf_b[idx], model_name='Baseline'
    )
    plot_grad_cam_row(
        axes[1], image, heatmap_a,
        CLASS_NAMES[y_true_a[idx]], CLASS_NAMES[y_pred_a[idx]],
        conf_a[idx], model_name='Augmentation'
    )

    plt.tight_layout(rect=[0, 0, 1, 0.90])
    output_path = PATHS['grad_cam'] / f'position_{pos}_idx{idx}.png'
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Gespeichert: {output_path}')